# Kaggriculture: Stage 2 - Opponent League

This notebook downloads six public agents, evaluates the current submission in both seats, checkpoints every game to Drive, and builds a Bradley-Terry ranking. It does **not** submit anything to Kaggle.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Kaggriculture')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Drive ready: {DRIVE_ROOT}', flush=True)

In [ ]:
import subprocess

REPOSITORY = 'https://github.com/GrigoriiIurev/Kaggriculture.git'
PROJECT = Path('/content/Kaggriculture')
if not (PROJECT / '.git').exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
import kaggle_environments
print(f'kaggle-environments: {kaggle_environments.__version__}', flush=True)

## Settings

Default mode runs 24 games. Full round robin runs 84 games. Completed games are reused after a restart.

In [ ]:
CHALLENGER = DRIVE_ROOT / 'results/rl_boatlee_v16/submission.tar.gz'
CHALLENGER_NAME = 'our_agent_1601'
SEED_COUNT = 2
SEED_START = 8100
FULL_ROUND_ROBIN = False
REFRESH_OPPONENTS = False
MAX_GAMES = 0  # 0 = all requested games; use 1 for a quick smoke run

if not CHALLENGER.is_file():
    candidates = sorted(DRIVE_ROOT.glob('**/submission.tar.gz'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError('No submission.tar.gz found on Drive')
    CHALLENGER = candidates[0]
print(f'Challenger: {CHALLENGER}', flush=True)

In [ ]:
command = [
    sys.executable, '-u', str(PROJECT / 'run_league_pipeline.py'),
    '--drive-root', str(DRIVE_ROOT),
    '--challenger', str(CHALLENGER),
    '--challenger-name', CHALLENGER_NAME,
    '--seed-count', str(SEED_COUNT),
    '--seed-start', str(SEED_START),
    '--max-games', str(MAX_GAMES),
]
if FULL_ROUND_ROBIN:
    command.append('--full-round-robin')
if REFRESH_OPPONENTS:
    command.append('--refresh-opponents')
LOG_PATH = DRIVE_ROOT / 'league/league_pipeline.log'
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Running:', ' '.join(command), flush=True)
print(f'Full log: {LOG_PATH}', flush=True)
with LOG_PATH.open('w', encoding='utf-8', buffering=1) as log_file:
    process = subprocess.Popen(
        command, cwd=PROJECT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
        log_file.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f'League pipeline failed with exit code {return_code}. Full error: {LOG_PATH}'
    )
print('League pipeline completed successfully.', flush=True)

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

RESULTS = DRIVE_ROOT / 'league/results'
display(pd.read_csv(RESULTS / 'rankings.csv'))
display(Markdown((RESULTS / 'report.md').read_text(encoding='utf-8')))
summary = json.loads((RESULTS / 'league_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary['promotion_gate'], indent=2), flush=True)